## DLT Pipeline

Streaming Table

In [0]:
#import dlt
#from pyspark.sql.functions import *
#from pyspark.sql.types import *
from pyspark import pipelines as dp
from pyspark.sql.functions import *
from pyspark.sql.types import *

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-5026261207729242>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import *
      3 from pyspark.sql.types import *

ModuleNotFoundError: No module named 'dlt'

In [0]:
#Expectations
rules = {
    "rule1":"product_id IS NOT NULL",
    "rule2":"product_name IS NOT NULL"
}

In [0]:
#@dlt.table()
#def DimProducts_stage():
#  df = spark.readStream.table("databricks035_catalog.products")
#  return df
@dp.table(name="DimProducts_stage")
@dp.expect_all_or_drop(rules)
def DimProducts_stage():

    return spark.readStream.option("skipChangeCommits", "true").table(
        "databricks035_catalog.silver.products"
    )

Streaming View

In [0]:
#@dlt.view
#def DimProducts_view():
#    df = spark.readStream.table("Live.DimProducts_stage")
#    return df
@dp.temporary_view
def DimProducts_view():
    df = spark.readStream.table("DimProducts_stage")
    return df

DimProducts

In [0]:
dp.create_streaming_table(name="DimProducts")


In [0]:
dp.create_auto_cdc_flow(
    target="DimProducts",
    source="DimProducts_stage",
    keys=["product_id"],
    sequence_by=col("product_id"),
    stored_as_scd_type="2"
)